# 00 — Pipeline Sprint 2 trên Google Colab (TrashCan-Instance, 22 lớp)

Notebook kiểm chứng quy trình Sprint 2 trên **dataset `v1_crop128` đã đóng băng**: khôi phục từ Drive → Data Audit → EDA → kiểm thử → huấn luyện E01, E02 → đánh giá Validation → demo → export kết quả lên Drive.

| Bước | Người phụ trách (dự phòng: Yến) | Thời gian ước tính (GPU T4) |
|---|---|---|
| 1–2. Cài đặt, khôi phục dataset | Hùng | 10–20 phút (Drive) |
| 3–4. Data Audit, EDA | Hùng | 5–10 phút |
| 5. Kiểm thử | Yến | 1 phút |
| 6–7. Huấn luyện E01, E02 | Bách | 10–25 phút mỗi run |
| 8. Đánh giá Validation | Ân | 2 phút |
| 9. Demo | Nguyên | 1 phút |
| 10. Báo cáo + export Drive | Yến | 2 phút |

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**. Quy ước và thông số: `docs/decisions/ADR-001-class-scope.md`, `configs/data_contract.yaml`. Tập **Test không được dùng** trong Sprint 2.

## 1. Cài đặt môi trường

In [ ]:
import os, sys, subprocess, time, json
from pathlib import Path

BRANCH = "fix/dataset-audit-mobilenetv2"  # kiểm chứng Draft PR #35
REPO_URL = "https://github.com/phanyen06ym-ien/ocean-plastic-ml-starter.git"
REPO = Path("/content/repo")

from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/ocean-plastic-ml")       # thư mục chung của nhóm trên Drive
for sub in ["data/raw", "data/processed", "runs"]:
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)

if not REPO.exists():
    subprocess.run(["git", "clone", "--single-branch", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    dirty = subprocess.run(["git", "-C", str(REPO), "status", "--porcelain"], capture_output=True, text=True, check=True).stdout.strip()
    if dirty:
        raise RuntimeError(f"{REPO} có thay đổi cục bộ. Hãy dùng runtime Colab sạch để kiểm chứng Draft PR; không tự xóa thay đổi.")
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth=1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
os.chdir(REPO); sys.path.insert(0, str(REPO))
assert subprocess.run(["git", "branch", "--show-current"], capture_output=True, text=True, check=True).stdout.strip() == BRANCH
COMMIT_SHA = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
assert (REPO / "src/data_audit.py").is_file(), "Checkout thiếu src/data_audit.py; kiểm tra nhánh PR"
subprocess.run(["git", "ls-files", "--error-unmatch", "src/data_audit.py"], capture_output=True, text=True, check=True)
print("PR branch:", BRANCH, "| commit SHA:", COMMIT_SHA, "| src/data_audit.py: present and tracked")

import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
print("Commit SHA:", COMMIT_SHA)

## 2. Dữ liệu TrashCan 1.0

Notebook chỉ dùng dataset đã xử lý `v1_crop128` đang lưu trên Drive. Nếu thư mục Colab chưa có, khôi phục từ ZIP trên Drive; nếu thiếu cả thư mục lẫn ZIP, notebook sẽ dừng và hướng dẫn khôi phục bản lưu.

Không tải TrashCan gốc, chạy lại preprocessing, đổi split/class mapping hoặc ghi mapping vào repository. Không commit dữ liệu lên GitHub.

## 3. Khôi phục và xác thực dataset đã đóng băng `v1_crop128` (không preprocessing)

In [ ]:
from src.common import load_contract
DATA_VERSION = load_contract()["dataset"]["data_version"]
assert DATA_VERSION == "v1_crop128", f"Expected frozen v1_crop128, got {DATA_VERSION}"
PROCESSED = Path("/content/processed") / DATA_VERSION
PROCESSED_ZIP = DRIVE / "data/processed" / f"{DATA_VERSION}.zip"
if not PROCESSED.exists():
    if not PROCESSED_ZIP.is_file():
        raise FileNotFoundError(f"Không thấy {PROCESSED} hoặc ZIP {PROCESSED_ZIP}. Hãy khôi phục `v1_crop128.zip` từ Drive vào MyDrive/ocean-plastic-ml/data/processed/, rồi chạy lại cell này. Không chạy preprocessing để tạo lại v1.")
    PROCESSED.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-q", str(PROCESSED_ZIP), "-d", str(PROCESSED.parent)], check=True)
    print("Đã khôi phục dataset đã đóng băng từ Drive:", PROCESSED_ZIP)
required = ["split_manifest.csv", "class_mapping.json", "preprocessing_summary.json", "class_split_counts.csv"]
missing = [name for name in required if not (PROCESSED / name).is_file()]
if missing:
    raise FileNotFoundError(f"Dataset đã khôi phục thiếu {missing}. Kiểm tra/khôi phục đầy đủ {PROCESSED_ZIP} từ Drive; không chạy preprocessing.")
summary = json.loads((PROCESSED / "preprocessing_summary.json").read_text(encoding="utf-8"))
assert summary["data_version"] == DATA_VERSION and summary["crops_per_split"] == {"train": 8283, "val": 1782, "test": 2009}, f"Unexpected/frozen v1 split counts: {summary}"
print("Dataset:", PROCESSED, "| split giữ nguyên:", summary["crops_per_split"])

In [ ]:
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
# Kiểm tra bắt buộc trước khi huấn luyện
import pandas as pd
from src.preprocessing import check_no_leakage
manifest = pd.read_csv(PROCESSED / "split_manifest.csv")
check_no_leakage(manifest)
print("Không rò rỉ video/ảnh giữa các tập ✔")
print("Số video:", summary["num_videos"], "(Dataset Audit của Hùng ghi nhận 312 — nếu chênh nhiều, kiểm tra regex video id)")
print("Lớp thiếu ở val:", summary["classes_missing_in"]["val"], "| thiếu ở test:", summary["classes_missing_in"]["test"])

## 4. EDA (S1/S2-DATA)

In [ ]:
from src import eda
from src.common import load_class_mapping, class_names
mapping = load_class_mapping(REPO / "configs/class_mapping.json")
NAMES = class_names(mapping)
PLASTIC = [m["class_name"] for m in mapping if m["plastic_related"]]
FIG, MET = REPO / "results/figures", REPO / "results/metrics"
FIG.mkdir(parents=True, exist_ok=True); MET.mkdir(parents=True, exist_ok=True)

dist = eda.class_distribution(manifest, by="source_split")
dist.to_csv(MET / "dataset_class_distribution.csv")
split_counts = pd.read_csv(PROCESSED / "class_split_counts.csv", index_col=0)
split_counts.to_csv(MET / "dataset_split_counts.csv")
eda.plot_class_distribution(dist, highlight=PLASTIC, save_path=FIG / "eda_class_distribution.png")
eda.plot_split_counts(split_counts, save_path=FIG / "eda_split_counts.png")
eda.plot_sample_crops(manifest, PROCESSED, NAMES, per_class=4, save_path=FIG / "eda_sample_crops.png")
print("Tỉ lệ mất cân bằng (lớn nhất/nhỏ nhất):", round(eda.imbalance_ratio(manifest), 1))
display(split_counts)
display(eda.bbox_stats(manifest))

### Data Audit trước huấn luyện

Audit chỉ đọc dataset `v1_crop128` đã có, kể cả Test để kiểm tra tính toàn vẹn; không đánh giá mô hình.
`CONDITIONAL` / `KEEP_V1_WITH_CONDITIONS` vẫn còn 3 điều kiện chưa xác minh (xem `docs/dataset_audit.md`).
Không chia lại dữ liệu, đổi mapping hoặc ghi đè checkpoint E03.


In [ ]:
from src.data_audit import run_audit

audit = run_audit(PROCESSED, DRIVE / "runs" / "data_audit_v1_crop128")
print(audit["status"], audit["decision"])
print("Pending:", audit["pending_verifications"])
assert audit["automated_checks_passed"], "Dataset integrity errors: inspect audit_summary.json before training"


## 5. Kiểm thử tự động (có TensorFlow)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-2000:])
assert r.returncode == 0, "Có test lỗi — dừng lại, báo Yến"

## 6. Huấn luyện E01 — CNN baseline + class weights

In [ ]:
# E01/E02: TensorFlow/Keras API. src.train is the E03 PyTorch pipeline.
from src.train_keras import load_experiment_config, run_training
RUNS = DRIVE / "runs"
cfg1 = load_experiment_config(REPO / "configs/experiments/E01_baseline.yaml")
t0 = time.time()
model1, hist1, RUN_E01 = run_training(cfg1, PROCESSED, RUNS)
print(f"E01 xong sau {(time.time()-t0)/60:.1f} phút → {RUN_E01}")
# Nếu Colab ngắt: chạy lại với run_training(cfg1, PROCESSED, RUNS, run_dir=<đường dẫn run cũ>, resume=True)

## 7. Huấn luyện E02 — E01 + Data Augmentation

In [ ]:
cfg2 = load_experiment_config(REPO / "configs/experiments/E02_augment.yaml")
t0 = time.time()
model2, hist2, RUN_E02 = run_training(cfg2, PROCESSED, RUNS)
print(f"E02 xong sau {(time.time()-t0)/60:.1f} phút → {RUN_E02}")

## 8. Đánh giá trên Validation (S2-EVAL) — KHÔNG dùng Test

In [ ]:
from src.evaluation_runner import evaluate_run, comparison_table
res = {}
for run in [RUN_E01, RUN_E02]:
    res[run.name] = evaluate_run(run, PROCESSED, split="val", results_dir=REPO / "results")
table = comparison_table(sorted((MET).glob("E0*_val.json")))
table.to_csv(MET / "comparison_val.csv", index=False)
display(table)
for k, r in res.items():
    print(k, "| Macro F1 22 lớp:", round(r["overall"]["macro_f1"], 4),
          "| Macro F1 nhóm nhựa:", r["plastic_related"]["macro_f1"] and round(r["plastic_related"]["macro_f1"], 4))

## 9. Demo dự đoán (S2-DEMO) với checkpoint tốt nhất theo Macro F1 Validation

In [ ]:
import matplotlib.pyplot as plt, numpy as np
from src.predict import Predictor
best_run = max([RUN_E01, RUN_E02], key=lambda r: res[r.name]["overall"]["macro_f1"])
print("Checkpoint demo:", best_run)
pred = Predictor(best_run / "best.keras", REPO / "configs/class_mapping.json")

val_ok = manifest[(manifest.split == "val") & (manifest.crop_status == "ok")]
pick = val_ok.sample(8, random_state=1)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, row in zip(axes.flat, pick.itertuples()):
    r = pred.predict_path(PROCESSED / row.crop_path, k=3)
    ax.imshow(plt.imread(PROCESSED / row.crop_path)); ax.axis("off")
    top = r["top1"]
    ok = "✔" if top["class_name"] == row.class_name else "✘"
    ax.set_title(f"{ok} thật: {row.class_name}\ndự đoán: {top['class_name']} ({top['confidence']:.0%})", fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "demo_val_predictions.png", dpi=110); plt.show()

sample_img = PROCESSED / pick.iloc[0].crop_path
print(subprocess.run([sys.executable, "main.py", "--image", str(sample_img), "--model", str(best_run / "best.keras")],
                     capture_output=True, text=True).stdout)

## 10. Báo cáo kết quả Sprint 2 và export lên Drive

Kết quả (metrics, hình, báo cáo, history) được chép vào Drive `exports/s2_results_<thời gian>/`. Khi cần đưa kết quả lên GitHub, tạo PR riêng từ Git clone đã xác thực an toàn; notebook không push hoặc xử lý token.

In [ ]:
def md_table(df):
    return df.to_markdown() if hasattr(df, "to_markdown") else df.to_string()

report = f'''# Kết quả Sprint 2 (tự sinh từ notebooks/00_pipeline_colab.ipynb)

- Ngày chạy: {time.strftime("%d/%m/%Y %H:%M")} · Commit code: {subprocess.run(["git","rev-parse","--short","HEAD"],capture_output=True,text=True).stdout.strip()}
- DATA_VERSION: `{DATA_VERSION}` · Drive: `{PROCESSED_ZIP}`
- Run E01: `{RUN_E01}` · Run E02: `{RUN_E02}`

## Dữ liệu
- Annotation: {summary["num_annotations"]} · ảnh nguồn: {summary["num_source_images"]} · video: {summary["num_videos"]}
- Trạng thái crop: {summary["crop_status"]}
- Crop theo tập: {summary["crops_per_split"]} · tỉ lệ ảnh nguồn: {summary["split"]["image_fraction"]}
- Lớp thiếu ở val: {summary["classes_missing_in"]["val"]} · thiếu ở test: {summary["classes_missing_in"]["test"]}

{md_table(split_counts)}

## So sánh E01 vs E02 (Validation)

{md_table(table.set_index("exp_id"))}

Chỉ số chính: Macro F1 trên 22 lớp; `plastic_macro_f1` = Macro F1 trên 6 lớp liên quan nhựa (ADR-001).
Lớp có support nhỏ cần diễn giải thận trọng. Tập Test chưa được sử dụng.
'''
(REPO / "docs/reports").mkdir(parents=True, exist_ok=True)
(REPO / "docs/reports/sprint2_results.md").write_text(report, encoding="utf-8")
print(report)

In [ ]:
# Đường cong huấn luyện, tóm tắt kiến trúc, chép kết quả lên Drive
import pandas as pd, matplotlib.pyplot as plt, shutil
for run in [RUN_E01, RUN_E02]:
    h = pd.read_csv(run / "history.csv")
    exp = run.name.split("_")[0]
    h.to_csv(MET / f"{exp}_history.csv", index=False)
    metrics = [c for c in h.columns if not c.startswith("val_") and f"val_{c}" in h.columns and c != "epoch"]
    fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 3.5))
    for ax, m in zip(np.atleast_1d(axes), metrics):
        ax.plot(h[m], label="train"); ax.plot(h[f"val_{m}"], label="val"); ax.set_title(f"{exp}: {m}"); ax.set_xlabel("epoch"); ax.legend()
    fig.tight_layout(); fig.savefig(FIG / f"{exp}_training_curves.png", dpi=110); plt.close(fig)
for name, m in [("E01", model1), ("E02", model2)]:
    lines = []; m.summary(print_fn=lambda s, **k: lines.append(s))
    (MET / f"{name}_model_summary.txt").write_text("\n".join(lines), encoding="utf-8")
shutil.copy(PROCESSED / "preprocessing_summary.json", MET / "preprocessing_summary.json")
eda.bbox_stats(manifest).to_csv(MET / "dataset_bbox_stats.csv")
EXPORT = DRIVE / "exports" / f"s2_results_{time.strftime('%m%d-%H%M')}"
for rel in ["results/metrics", "results/figures"]:
    shutil.copytree(REPO / rel, EXPORT / rel, dirs_exist_ok=True)
for rel in ["configs/class_mapping.json", "docs/reports/sprint2_results.md"]:
    (EXPORT / rel).parent.mkdir(parents=True, exist_ok=True); shutil.copy(REPO / rel, EXPORT / rel)
(EXPORT / "runs.json").write_text(json.dumps({"E01": str(RUN_E01), "E02": str(RUN_E02), "best": str(best_run)}, indent=2))
print("Đã lưu kết quả lên Drive:", EXPORT)
print("\n".join(str(p.relative_to(EXPORT)) for p in sorted(EXPORT.rglob("*")) if p.is_file()))

K?t qu? ?? ???c export l?n Drive ? cell tr?n. N?u c?n chia s? tr?n GitHub, t?o m?t PR ri?ng t? Git clone ?? x?c th?c an to?n; notebook kh?ng push GitHub v? kh?ng nh?n token.
